In [5]:
import zipfile
import os
zip_path = "archive.zip"      # your zip file name
extract_path = "brain_tumor_dataset" # folder to extract into
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [19]:
!pip install scikit-learn tensorflow matplotlib scipy

In [22]:
import os
import numpy as np
import matplotlib.pyplot as plt

In [23]:
train_path = "brain_tumor_dataset/BrainTumor_1/Train"
test_path  = "brain_tumor_dataset/BrainTumor_1/Test"
print(train_path)
print(test_path)

brain_tumor_dataset/BrainTumor_1/Train
brain_tumor_dataset/BrainTumor_1/Test


In [24]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True)
# Only rescaling for testing/validation
test_datagen = ImageDataGenerator(rescale=1./255)
# Load training data
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical')
# Load validation data
val_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 22848 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()
# Block 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(MaxPooling2D(2,2))
# Block 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))
# Block 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))
# Flatten
model.add(Flatten())
# Fully Connected Layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
# Output Layer (4 classes)
model.add(Dense(train_data.num_classes, activation='softmax'))
# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True)

In [32]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=14,
    callbacks=[early_stop])

Epoch 1/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 2085s 3s/step - accuracy: 0.6370 - loss: 0.8660 - val_accuracy: 0.6720 - val_loss: 0.7830
Epoch 2/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 752s 1s/step - accuracy: 0.7709 - loss: 0.5903 - val_accuracy: 0.7788 - val_loss: 0.5476
Epoch 3/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 598s 837ms/step - accuracy: 0.8108 - loss: 0.4904 - val_accuracy: 0.8047 - val_loss: 0.4710
Epoch 4/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 577s 807ms/step - accuracy: 0.8417 - loss: 0.4201 - val_accuracy: 0.8337 - val_loss: 0.4165
Epoch 5/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 576s 807ms/step - accuracy: 0.8614 - loss: 0.3742 - val_accuracy: 0.8505 - val_loss: 0.4355
Epoch 6/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 710s 994ms/step - accuracy: 0.8755 - loss: 0.3411 - val_accuracy: 0.8337 - val_loss: 0.4451
Epoch 7/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 601s 841ms/step - accuracy: 0.8873 - loss: 0.3056 - val_accuracy: 0.8741 - val_loss: 0.3134
Epoch 8/14
714/714 ━━━━━━━━━━━━━━━━━━━━ 603s 844ms/step - accuracy: 0.8950 - loss

In [34]:
model.save("brain_tumor_model.keras")